In [ ]:
import requests
import pandas as pd
import time
import re

def kriter_ayikla(metin, anahtar_kelimeler):
    """Metin içinde geçen belirli becerileri bulur."""
    metin = str(metin).lower()
    bulunanlar = [kelime for kelime in anahtar_kelimeler if re.search(rf'\b{kelime}\b', metin)]
    return ", ".join(bulunanlar)

def veri_cek_ve_analiz_et(hedef=10000):
    all_jobs = []
    
    teknik_beceriler = ['python', 'sql', 'java', 'aws', 'docker', 'kubernetes', 'excel', 'tableau', 'power bi']
    sosyal_beceriler = ['communication', 'leadership', 'teamwork', 'problem solving', 'management', 'agile']
    diller = ['english', 'german', 'french', 'spanish']
    
    print(f"Himalayas üzerinden {hedef} veri toplama ve analiz başlatıldı...")
    
    offset = 0
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    while len(all_jobs) < hedef:
        url = f"https://himalayas.app/jobs/api?limit=100&offset={offset}"
        try:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code == 200:
                data = response.json()
                jobs = data.get('jobs', [])
                if not jobs: break
                
                for j in jobs:
                    aciklama_ham = j.get('description', '')
                    aciklama_temiz = re.sub(r'<[^>]*>', '', aciklama_ham)
                    
                    tek_beceri = kriter_ayikla(aciklama_temiz, teknik_beceriler)
                    sos_beceri = kriter_ayikla(aciklama_temiz, sosyal_beceriler)
                    dil_beceri = kriter_ayikla(aciklama_temiz, diller)
                    
                    all_jobs.append({
                        'Unvan': j.get('title'),
                        'Sirket': j.get('company_name'),
                        'Lokasyon': j.get('location'),
                        'Teknik_Beceriler': tek_beceri,
                        'Sosyal_Beceriler': sos_beceri,
                        'Dil_Kriteri': dil_beceri,
                        'Tum_Kriterler': ", ".join(filter(None, [tek_beceri, sos_beceri, dil_beceri])),
                        'Aciklama': aciklama_temiz
                    })
                
                print(f"İlerleme: {len(all_jobs)} / {hedef}")
                offset += 100
                time.sleep(0.5)
            else:
                break
        except Exception as e:
            print(f"Hata: {e}")
            break

    df = pd.DataFrame(all_jobs)
    
    df['Beceri_Sayisi'] = df['Tum_Kriterler'].apply(lambda x: len(x.split(', ')) if x else 0)
    
    df.to_csv("ik_yetenek_analizi_veriseti.csv", index=False, encoding='utf-8-sig')
    return df

df_final = veri_cek_ve_analiz_et(10000)